# U1 - Integrante 3 - Clasificación de Rol (is_buyer_maker)

Pipeline Bronze → Silver → Gold con PySpark, entrenamiento y comparación de modelos de clasificación binaria (Logistic Regression vs GBT Classifier).

## 1. Configuración Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, BooleanType
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = (
    SparkSession.builder
        .appName("U1-Integrante3-ClasificacionRol")
        .master("local[*]")
        .config("spark.ui.port", "4040")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 15:14:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Carga Bronze

In [2]:
schema = StructType([
    StructField("trade_id", LongType(), False),
    StructField("price", DoubleType(), True),
    StructField("qty", DoubleType(), True),
    StructField("quote_qty", DoubleType(), True),
    StructField("time_us", LongType(), False),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True)
])

df_bronze = spark.read.csv("/opt/data/BTCUSDT-trades-2026-01-05.csv", schema=schema, header=False)
df_bronze.printSchema()
df_bronze.show(5)

root
 |-- trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- quote_qty: double (nullable = true)
 |-- time_us: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)

+----------+--------+-------+-----------+----------------+--------------+-------------+
|  trade_id|   price|    qty|  quote_qty|         time_us|is_buyer_maker|is_best_match|
+----------+--------+-------+-----------+----------------+--------------+-------------+
|5734054604|91529.74| 2.2E-4| 20.1365428|1767571200308618|         false|         true|
|5734054605|91529.74|   0.01|   915.2974|1767571200375801|         false|         true|
|5734054606|91529.74|0.00437|399.9849638|1767571200477157|         false|         true|
|5734054607|91529.74|0.00764|699.2872136|1767571200480543|         false|         true|
|5734054608|91529.74|0.00136|124.4804464|1767571200545508|         false|         true|
+--------

## 3. Capa Silver

In [3]:
df_silver = df_bronze.na.drop(subset=["trade_id", "price", "is_buyer_maker"]) \
    .dropDuplicates(["trade_id"]) \
    .withColumn("label", when(col("is_buyer_maker") == True, 1.0).otherwise(0.0))

df_silver.show(5)

[Stage 1:>                                                        (0 + 10) / 10]

+----------+--------+-------+-------------+----------------+--------------+-------------+-----+
|  trade_id|   price|    qty|    quote_qty|         time_us|is_buyer_maker|is_best_match|label|
+----------+--------+-------+-------------+----------------+--------------+-------------+-----+
|5734054604|91529.74| 2.2E-4|   20.1365428|1767571200308618|         false|         true|  0.0|
|5734054610|91529.74| 5.4E-4|   49.4260596|1767571200665868|         false|         true|  0.0|
|5734054624|91529.74|0.20378|18651.9304172|1767571202033686|         false|         true|  0.0|
|5734054642|91529.73|0.00599|  548.2630827|1767571202551395|          true|         true|  1.0|
|5734054646|91529.74| 9.0E-5|    8.2376766|1767571202941974|         false|         true|  0.0|
+----------+--------+-------+-------------+----------------+--------------+-------------+-----+
only showing top 5 rows


## 4. Capa Gold Parquet

In [4]:
df_silver.write.mode("overwrite").partitionBy("is_best_match").parquet("/opt/artifacts/gold_integrante3")
df_gold = spark.read.parquet("/opt/artifacts/gold_integrante3")
df_gold.show(5)

+----------+--------+-------+-----------+----------------+--------------+-----+-------------+
|  trade_id|   price|    qty|  quote_qty|         time_us|is_buyer_maker|label|is_best_match|
+----------+--------+-------+-----------+----------------+--------------+-----+-------------+
|5734054606|91529.74|0.00437|399.9849638|1767571200477157|         false|  0.0|         true|
|5734054617|91529.74| 9.6E-4| 87.8685504|1767571201602432|         false|  0.0|         true|
|5734054630|91529.74| 1.1E-4| 10.0682714|1767571202085489|         false|  0.0|         true|
|5734054637|91529.73| 4.7E-4| 43.0189731|1767571202435549|          true|  1.0|         true|
|5734054644|91529.73|0.00476|435.6815148|1767571202744883|          true|  1.0|         true|
+----------+--------+-------+-----------+----------------+--------------+-----+-------------+
only showing top 5 rows


## 5. VectorAssembler & Scaler

In [5]:
assembler = VectorAssembler(inputCols=["price", "qty", "quote_qty"], outputCol="features")
df_ml = assembler.transform(df_gold).select("features", "label")

df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

## 6. Modelado Comparativo

In [6]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
modelo_lr = lr.fit(df_train)
pred_lr = modelo_lr.transform(df_test)

gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=10)
modelo_gbt = gbt.fit(df_train)
pred_gbt = modelo_gbt.transform(df_test)

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
                                                                                

## 7. Evaluación AUC

In [7]:
evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
print(f"Logistic Regression AUC: {evaluator.evaluate(pred_lr):.4f}")
print(f"GBT Classifier AUC: {evaluator.evaluate(pred_gbt):.4f}")

Logistic Regression AUC: 0.5299


GBT Classifier AUC: 0.5578


## 8. Guardar Artefacto Ganador

In [8]:
modelo_gbt.write().overwrite().save("/opt/artifacts/modelo_ganador_integrante3")